# GrapePPI Prediction

Use a trained GrapePPI model to predict protein–protein interactions. Input TSV: protein1_ID, protein2_ID (optional third column: label).

## 1. Dependencies and Model Definition

In [ ]:
# Google Colab environment setup (run first if using Colab)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Set to your Drive folder containing data/ and models/, or "/content" if you upload files
    COLAB_PROJECT_ROOT = "/content/drive/MyDrive/grape-ppi"
    import os
    os.chdir(COLAB_PROJECT_ROOT)
    print(f"Colab: cwd = {os.getcwd()}")
else:
    COLAB_PROJECT_ROOT = None
    print("Running locally.")

In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


class GrapePPI(nn.Module):
    def __init__(self, esm_dim=1280, hidden_dim=512, dropout=0.2):
        super(GrapePPI, self).__init__()
        self.sequence_encoder = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.interaction_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
        )

    def combine_features(self, x1, x2):
        return torch.cat([x1, x2], dim=-1)

    def forward(self, x1, x2):
        x1 = self.sequence_encoder(x1)
        x2 = self.sequence_encoder(x2)
        x = self.combine_features(x1, x2)
        logits = self.interaction_predictor(x)
        return F.sigmoid(logits).squeeze()

In [ ]:
class PooledDataset(Dataset):
    def __init__(self, names1, names2, labels, embedding_h5):
        self.names1 = names1
        self.names2 = names2
        self.labels = labels
        self.embed_data = {}
        names = set(names1).union(set(names2))
        with h5py.File(embedding_h5, "r") as h5fin:
            for name in names:
                self.embed_data[name] = torch.from_numpy(h5fin[name][:]).float()

    def __len__(self):
        return len(self.names1)

    def __getitem__(self, i):
        x1 = self.embed_data[self.names1[i]]
        x2 = self.embed_data[self.names2[i]]
        return x1, x2, torch.as_tensor(self.labels[i]).float()


def load_pooled_data(data_file, batch_size, embedding_h5, train=False):
    df = pd.read_csv(data_file, sep="\t", header=None)
    n_cols = df.shape[1]
    names1 = df[0].to_list()
    names2 = df[1].to_list()
    labels = df[2].to_list() if n_cols >= 3 else [0.0] * len(names1)
    dataset = PooledDataset(names1, names2, labels, embedding_h5)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False), names1, names2, labels

## 2. Configuration: Model Path and Data Path

In [ ]:
# For Grape-1 dataset
data_dir = "./data/Grape-1"
model_path = os.path.join("models", "grape_1_model.pt")

# For Grape-10 dataset
# data_dir = "./data/Grape-10"
# model_path = os.path.join("models", "grape_10_model.pt")

embedding_h5 = os.path.join(data_dir, "esm2_t36_3B.h5")
# TSV to predict: at least two columns (protein1_ID, protein2_ID), optional third column for ground-truth label
predict_tsv = os.path.join(data_dir, "test.tsv")
output_csv = os.path.join("predictions", "test_prediction.csv")

batch_size = 128
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Model: {model_path}")
print(f"Input: {predict_tsv}")
print(f"Output: {output_csv}")

## 3. Load Model and Predict

In [ ]:
ckpt = torch.load(model_path, map_location=device, weights_only=False)
esm_dim = ckpt.get("esm_dim", 2560)
model = GrapePPI(esm_dim=esm_dim, hidden_dim=512, dropout=0.2)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device)
model.eval()
print(f"Loaded epoch {ckpt.get('epoch', '?')}, val_loss={ckpt.get('val_loss', 0.):.4f}")
print(f"esm_dim={esm_dim}")

In [ ]:
loader, names1, names2, labels = load_pooled_data(predict_tsv, batch_size, embedding_h5)
y_pred_list = []
y_true_list = []
with torch.no_grad():
    for x1, x2, y in loader:
        x1, x2 = x1.to(device), x2.to(device)
        pred = model(x1, x2)
        y_pred_list.append(pred.cpu().numpy())
        y_true_list.append(y.numpy())

y_pred = np.concatenate(y_pred_list)
y_true = np.concatenate(y_true_list)
print(f"Predictions: {len(y_pred)}")

## 4. Save Prediction Results

In [ ]:
os.makedirs(os.path.dirname(output_csv) or '.', exist_ok=True)
out_df = pd.DataFrame({
    "protein1": names1,
    "protein2": names2,
    "y_pred": y_pred,
    "y_true": y_true,
})
out_df.to_csv(output_csv, index=False)
print(f"Saved to {output_csv}")
out_df.head(10)